# Notebook 5: LLM Optimization & Serving Concepts

**Goal:** Master the engineering side of LLM deployment. Mistral's system design round often asks about inference optimization.

**Topics:**
1. GPU memory math (how much memory does a model need?)
2. KV cache — simulation and memory analysis
3. Quantization — INT8 and INT4 concepts with code
4. Continuous batching simulation
5. Throughput vs latency analysis
6. Speculative decoding concept
7. Flash Attention motivation
8. Parallelism strategies
9. Memory-efficient inference patterns
10. System design: serving a 70B model

In [ ]:
import numpy as np
import math
import time
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from collections import deque

print('Setup complete')

---
## 1. GPU Memory Math

**Know these numbers cold for interviews!**

Memory for model weights:
- FP32: 4 bytes/param
- BF16/FP16: 2 bytes/param
- INT8: 1 byte/param
- INT4: 0.5 bytes/param

Memory for KV cache (per token, per layer):
- `2 * num_kv_heads * head_dim * dtype_bytes`
- The 2 is for K and V

In [ ]:
@dataclass
class ModelConfig:
    name: str
    num_params: int           # total parameter count
    num_layers: int
    d_model: int              # hidden dimension
    num_q_heads: int
    num_kv_heads: int         # GQA: may be < num_q_heads
    head_dim: int
    vocab_size: int
    max_seq_len: int


MISTRAL_7B = ModelConfig(
    name='Mistral-7B',
    num_params=7_300_000_000,
    num_layers=32,
    d_model=4096,
    num_q_heads=32,
    num_kv_heads=8,           # GQA
    head_dim=128,
    vocab_size=32000,
    max_seq_len=32768,
)

MIXTRAL_8X7B = ModelConfig(
    name='Mixtral-8x7B',
    num_params=46_700_000_000,  # total, but only ~13B active
    num_layers=32,
    d_model=4096,
    num_q_heads=32,
    num_kv_heads=8,
    head_dim=128,
    vocab_size=32000,
    max_seq_len=32768,
)


def model_memory_gb(
    config: ModelConfig,
    dtype: str = 'bf16',
) -> float:
    """
    Estimate model weight memory in GB.
    dtype options: 'fp32', 'fp16', 'bf16', 'int8', 'int4'
    """
    bytes_per_param = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1, 'int4': 0.5}
    # YOUR CODE HERE
    raise NotImplementedError()


def kv_cache_memory_gb(
    config: ModelConfig,
    batch_size: int,
    seq_len: int,
    dtype: str = 'bf16',
) -> float:
    """
    KV cache memory = 2 (K+V) * num_layers * num_kv_heads * head_dim * batch * seq_len * bytes
    """
    bytes_per_element = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1, 'int4': 0.5}
    # YOUR CODE HERE
    raise NotImplementedError()


# Test: should give ~14GB for 7B BF16
# print(f'Mistral 7B BF16: {model_memory_gb(MISTRAL_7B, "bf16"):.1f} GB')
# print(f'Mistral 7B INT4: {model_memory_gb(MISTRAL_7B, "int4"):.1f} GB')

In [ ]:
# REFERENCE
def model_memory_gb_ref(config: ModelConfig, dtype: str = 'bf16') -> float:
    bytes_per_param = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1, 'int4': 0.5}
    bpp = bytes_per_param[dtype]
    return (config.num_params * bpp) / (1024 ** 3)


def kv_cache_memory_gb_ref(config: ModelConfig, batch_size: int, seq_len: int, dtype: str = 'bf16') -> float:
    bytes_per_element = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1, 'int4': 0.5}
    bpe = bytes_per_element[dtype]
    # 2 = K and V, each of shape (batch, num_kv_heads, seq_len, head_dim)
    total_bytes = 2 * config.num_layers * config.num_kv_heads * config.head_dim * batch_size * seq_len * bpe
    return total_bytes / (1024 ** 3)


model_memory_gb = model_memory_gb_ref
kv_cache_memory_gb = kv_cache_memory_gb_ref

print('=== MISTRAL 7B MEMORY ANALYSIS ===')
print(f'Weights BF16:    {model_memory_gb(MISTRAL_7B, "bf16"):.1f} GB')
print(f'Weights INT8:    {model_memory_gb(MISTRAL_7B, "int8"):.1f} GB')
print(f'Weights INT4:    {model_memory_gb(MISTRAL_7B, "int4"):.1f} GB')
print()
print('KV Cache (batch=1):')
for seq_len in [1024, 4096, 16384, 32768]:
    kv = kv_cache_memory_gb(MISTRAL_7B, batch_size=1, seq_len=seq_len)
    print(f'  seq_len={seq_len:5d}: {kv:.2f} GB')

print()
print('=== TOTAL ON A 24GB GPU (RTX 4090) ===')
weights = model_memory_gb(MISTRAL_7B, 'bf16')
kv = kv_cache_memory_gb(MISTRAL_7B, batch_size=4, seq_len=4096)
print(f'Weights ({weights:.1f}GB) + KV Cache batch=4, seq=4096 ({kv:.1f}GB) = {weights+kv:.1f}GB')
print(f'GPU headroom (24GB): {24 - weights - kv:.1f}GB')

---
## 2. KV Cache Simulation

**Why KV cache?**
Without it: for each new token, we recompute K and V for ALL previous tokens → O(n²) total compute.
With it: we cache K and V, only compute for the new token → O(n) total compute.

**What's cached:** K and V matrices for each layer at each sequence position.
**NOT cached:** Q — because Q depends on the current token only and must change each step.

In [ ]:
@dataclass
class KVCache:
    """
    Simulates a KV cache for autoregressive generation.
    In real implementations (vLLM), uses PagedAttention for fragmented memory.
    """
    num_layers: int
    num_kv_heads: int
    head_dim: int
    max_seq_len: int
    
    def __post_init__(self):
        # Pre-allocate cache: (num_layers, 2, max_seq_len, num_kv_heads, head_dim)
        # 2 = K and V
        self.cache = np.zeros((self.num_layers, 2, self.max_seq_len, self.num_kv_heads, self.head_dim))
        self.current_len = 0  # how many tokens are cached
    
    def update(self, layer: int, key: np.ndarray, value: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Store new K, V at current_len position and return full K, V tensors.
        
        key, value: (num_kv_heads, 1, head_dim) — single new token
        Returns: (full_keys, full_values) each (num_kv_heads, current_len+1, head_dim)
        """
        # YOUR CODE HERE
        # 1. Write key/value to cache at self.current_len (only for layer 0 update counter)
        # 2. Return the full cached sequence up to current position
        raise NotImplementedError()
    
    def finalize_step(self):
        """Called after all layers are updated for a single token."""
        self.current_len += 1
    
    @property
    def memory_mb(self) -> float:
        """Current memory used by cache in MB."""
        # YOUR CODE HERE
        raise NotImplementedError()


# Test
cache = KVCache(num_layers=32, num_kv_heads=8, head_dim=128, max_seq_len=4096)
print('Initial cache size:', cache.current_len)
print('Pre-allocated:', cache.cache.nbytes / 1024**2, 'MB')

In [ ]:
# REFERENCE
@dataclass
class KVCacheRef:
    num_layers: int
    num_kv_heads: int
    head_dim: int
    max_seq_len: int
    
    def __post_init__(self):
        self.cache = np.zeros((self.num_layers, 2, self.max_seq_len, self.num_kv_heads, self.head_dim), dtype=np.float16)
        self.current_len = 0
    
    def update(self, layer: int, key: np.ndarray, value: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        # key, value shape: (num_kv_heads, 1, head_dim)
        # Store at current position
        self.cache[layer, 0, self.current_len] = key[:, 0, :]  # K
        self.cache[layer, 1, self.current_len] = value[:, 0, :]  # V
        
        # Return full cached sequence
        end = self.current_len + 1
        full_k = self.cache[layer, 0, :end].transpose(1, 0, 2)  # (seq, heads, dim) -> back to whatever
        full_v = self.cache[layer, 1, :end].transpose(1, 0, 2)
        return full_k, full_v
    
    def finalize_step(self):
        self.current_len += 1
    
    @property
    def memory_mb(self) -> float:
        used_elements = self.num_layers * 2 * self.current_len * self.num_kv_heads * self.head_dim
        return (used_elements * 2) / (1024 ** 2)  # FP16 = 2 bytes


KVCache = KVCacheRef

# Simulate generation: show KV cache growing
cache = KVCache(num_layers=32, num_kv_heads=8, head_dim=128, max_seq_len=4096)

print('KV Cache growth during generation:')
for step in range(0, 200, 50):
    # Simulate inserting 50 tokens
    for _ in range(50):
        for layer in range(cache.num_layers):
            k = np.random.randn(cache.num_kv_heads, 1, cache.head_dim).astype(np.float16)
            v = np.random.randn(cache.num_kv_heads, 1, cache.head_dim).astype(np.float16)
            cache.update(layer, k, v)
        cache.finalize_step()
    print(f'  Tokens: {cache.current_len:4d} | KV Cache: {cache.memory_mb:.1f} MB')

---
## 3. Quantization

**Idea:** Represent model weights in lower precision (INT8, INT4) to save memory and speed up matrix multiplications.

**Techniques:**
- **GPTQ:** Post-training quantization using second-order Hessian approximation
- **AWQ:** Activation-aware weight quantization — identifies important channels
- **bitsandbytes (bnb):** INT8/INT4 with absmax or NF4 quantization
- **GGUF:** CPU-friendly format used by llama.cpp

In [ ]:
def absmax_quantize_int8(weights: np.ndarray) -> Tuple[np.ndarray, float]:
    """
    Absmax INT8 quantization.
    Maps weights to [-128, 127] based on max absolute value.
    
    Returns:
        quantized: int8 array
        scale: float scale factor to reconstruct
    """
    # YOUR CODE HERE
    # scale = max(|w|) / 127
    # q = round(w / scale)
    # clip to [-128, 127]
    raise NotImplementedError()


def dequantize_int8(quantized: np.ndarray, scale: float) -> np.ndarray:
    """
    Reconstruct weights from INT8: w_approx = quantized * scale
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# Test
np.random.seed(42)
weights = np.random.randn(100, 100).astype(np.float32)
# q_weights, scale = absmax_quantize_int8(weights)
# w_restored = dequantize_int8(q_weights, scale)
# error = np.abs(weights - w_restored).mean()
# print(f'INT8 quantization error (mean abs): {error:.6f}')
# print(f'Memory reduction: {weights.nbytes} -> {q_weights.nbytes} bytes ({q_weights.nbytes/weights.nbytes:.1%})')

In [ ]:
# REFERENCE
def absmax_quantize_int8_ref(weights: np.ndarray) -> Tuple[np.ndarray, float]:
    scale = np.abs(weights).max() / 127.0
    quantized = np.round(weights / scale).astype(np.int8)
    quantized = np.clip(quantized, -128, 127)
    return quantized, float(scale)


def dequantize_int8_ref(quantized: np.ndarray, scale: float) -> np.ndarray:
    return quantized.astype(np.float32) * scale


absmax_quantize_int8 = absmax_quantize_int8_ref
dequantize_int8 = dequantize_int8_ref

np.random.seed(42)
weights = np.random.randn(100, 100).astype(np.float32)
q_weights, scale = absmax_quantize_int8(weights)
w_restored = dequantize_int8(q_weights, scale)
error = np.abs(weights - w_restored).mean()
print(f'INT8 quantization:')
print(f'  Mean absolute error: {error:.6f}')
print(f'  Memory: {weights.nbytes}B -> {q_weights.nbytes}B ({q_weights.nbytes/weights.nbytes:.0%})')
print(f'  Scale factor: {scale:.6f}')

# Compare INT8 vs INT4 (simulated)
def quantize_int4_sim(weights: np.ndarray) -> Tuple[np.ndarray, float]:
    """Simulate INT4 (range: -8 to 7)"""
    scale = np.abs(weights).max() / 7.0
    quantized = np.round(weights / scale).clip(-8, 7).astype(np.int8)
    return quantized, float(scale)

q4, s4 = quantize_int4_sim(weights)
w4 = q4.astype(np.float32) * s4
print(f'\nINT4 (simulated):')
print(f'  Mean absolute error: {np.abs(weights - w4).mean():.6f}')  # higher error

---
## 4. Throughput vs Latency Analysis

In [ ]:
def analyze_serving_tradeoffs(
    model_tps_per_request: float = 50.0,     # tokens/sec for batch_size=1
    tokens_per_request: int = 200,           # avg output length
    max_batch_size: int = 32,
    # Batch processing: throughput scales with batch size, latency increases
    throughput_scale: float = 0.9,           # efficiency factor per batch doubling
):
    """
    Model serving throughput/latency tradeoffs.
    """
    print('Batch Size | Throughput (req/s) | Latency (s/req) | TTFT (ms) | GPU Util %')
    print('-' * 80)
    
    for bs in [1, 2, 4, 8, 16, 32]:
        if bs > max_batch_size:
            break
        
        # YOUR CODE HERE
        # Throughput: batch processing amortizes fixed costs
        # Latency: larger batches = more queueing + slower per-batch compute
        # Simplistic model: just calculate these values
        
        # Effective TPS for the batch (near-linear with some efficiency loss)
        batch_efficiency = throughput_scale ** math.log2(bs)
        total_tps = model_tps_per_request * bs * batch_efficiency
        throughput = total_tps / tokens_per_request
        latency = tokens_per_request / (total_tps / bs)
        ttft = (1 / (total_tps / bs)) * 1000  # ms for first token
        gpu_util = min(100, bs * 3.1)
        
        print(f'{bs:10d} | {throughput:18.1f} | {latency:15.2f} | {ttft:9.1f} | {gpu_util:10.1f}')

analyze_serving_tradeoffs()

---
## 5. Continuous Batching Simulation

**Static batching:** Wait for a full batch, process together. Problem: short requests wait for long ones.

**Continuous batching (PagedAttention/vLLM):** Process tokens from multiple requests in the same forward pass. When one finishes, immediately add a new one.

**Why it matters:** GPU utilization goes from ~30% (static) to ~80%+ (continuous).

In [ ]:
@dataclass
class Request:
    id: int
    prompt_len: int
    max_output_len: int
    arrival_time: float
    tokens_generated: int = 0
    start_time: Optional[float] = None
    finish_time: Optional[float] = None
    
    @property
    def is_done(self) -> bool:
        return self.tokens_generated >= self.max_output_len
    
    @property
    def latency(self) -> Optional[float]:
        if self.finish_time and self.start_time:
            return self.finish_time - self.arrival_time
        return None


def simulate_static_batching(
    requests: List[Request],
    batch_size: int = 4,
    tokens_per_second: float = 200.0,  # GPU throughput
) -> Dict:
    """
    Static batching: collect full batch, process until all done, then next batch.
    """
    # YOUR CODE HERE: simulate static batching
    # Process in batches of batch_size
    # Each batch takes (max output len in batch) / tokens_per_second seconds
    raise NotImplementedError()


def simulate_continuous_batching(
    requests: List[Request],
    max_concurrent: int = 4,
    tokens_per_second_per_slot: float = 50.0,
) -> Dict:
    """
    Continuous batching: always fill slots, swap in new requests when slots free.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

print('Batching simulators defined — implement above')

In [ ]:
# REFERENCE — static batching
def simulate_static_batching_ref(
    requests: List[Request],
    batch_size: int = 4,
    tokens_per_second: float = 200.0,
) -> Dict:
    import copy
    reqs = [copy.deepcopy(r) for r in requests]
    current_time = 0.0
    completed = []
    
    for i in range(0, len(reqs), batch_size):
        batch = reqs[i:i+batch_size]
        max_output = max(r.max_output_len for r in batch)
        batch_time = max_output / tokens_per_second
        
        for r in batch:
            r.start_time = current_time
            r.tokens_generated = r.max_output_len
            r.finish_time = current_time + batch_time
            completed.append(r)
        
        current_time += batch_time
    
    latencies = [r.latency for r in completed]
    return {
        'strategy': 'static',
        'total_time': current_time,
        'avg_latency': np.mean(latencies),
        'p95_latency': np.percentile(latencies, 95),
        'throughput_rps': len(reqs) / current_time,
    }


simulate_static_batching = simulate_static_batching_ref

# Generate test requests with mixed output lengths
np.random.seed(42)
test_requests = [
    Request(
        id=i,
        prompt_len=np.random.randint(10, 100),
        max_output_len=np.random.randint(10, 500),  # mixed lengths!
        arrival_time=0.0,
    )
    for i in range(20)
]

static_stats = simulate_static_batching(test_requests, batch_size=4)
print('Static Batching:')
for k, v in static_stats.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.3f}')
    else:
        print(f'  {k}: {v}')

---
## 6. Flash Attention — Motivation

**Problem:** Standard attention materializes the full (seq_len × seq_len) attention matrix.
For seq_len=32k: 32k × 32k = 1B elements × 2 bytes = 2GB just for ONE attention matrix!

**Flash Attention solution:** Tile the computation to fit in SRAM (fast on-chip memory).
- Reads/writes to HBM (GPU RAM) happen in tiles, not the full matrix
- Same numerical result, but IO complexity goes from O(N²) to O(N²/M) where M = SRAM size
- Typically 2-4x faster in practice

In [ ]:
def memory_usage_standard_attention(seq_len: int, d_model: int, num_heads: int) -> Dict[str, float]:
    """
    Calculate memory usage for standard vs flash attention.
    """
    d_k = d_model // num_heads
    
    # Standard attention: must store full N×N score matrix
    # For ALL heads in a layer
    attn_matrix_bytes = seq_len * seq_len * num_heads * 2  # FP16
    qkv_bytes = 3 * seq_len * d_model * 2  # Q, K, V
    output_bytes = seq_len * d_model * 2
    
    return {
        'seq_len': seq_len,
        'attn_matrix_mb': attn_matrix_bytes / 1e6,
        'qkv_mb': qkv_bytes / 1e6,
        'total_mb': (attn_matrix_bytes + qkv_bytes + output_bytes) / 1e6,
    }


print('Standard Attention Memory (Mistral 7B config, 32 heads, d_model=4096):')
print(f'{"Seq Len":>10} | {"Attn Matrix (MB)":>18} | {"Total (MB)":>12}')
print('-' * 45)
for seq_len in [512, 1024, 4096, 8192, 16384, 32768]:
    stats = memory_usage_standard_attention(seq_len, d_model=4096, num_heads=32)
    print(f'{seq_len:>10} | {stats["attn_matrix_mb"]:>18.1f} | {stats["total_mb"]:>12.1f}')

print('\nFlash Attention: O(1) memory for attention matrix (tiles fit in SRAM)')

---
## 7. Parallelism Strategies

How to distribute a large model across multiple GPUs:

| Strategy | What's split | When to use |
|---|---|---|
| Data Parallelism | Input batch | Training, inference with many small requests |
| Tensor Parallelism | Weight matrices (column/row) | Model too big for one GPU |
| Pipeline Parallelism | Layers (pipeline stages) | Very deep models, bandwidth limited |
| Expert Parallelism | MoE experts | Mixtral-style models |

**Mistral interview Q:** *If you have a 70B model and 8 × 80GB GPUs, how would you shard it?*

In [ ]:
def tensor_parallel_linear(
    x: np.ndarray,     # (batch, in_features)
    W: np.ndarray,     # (in_features, out_features)
    num_gpus: int,
    mode: str = 'column',  # 'column' or 'row'
) -> np.ndarray:
    """
    Simulate tensor parallelism for a linear layer.
    
    Column parallel: split W along output dim (each GPU gets slice of output)
    Row parallel: split W along input dim (each GPU gets slice of input)
    """
    # YOUR CODE HERE
    # Split W across GPUs, compute partial results, then all-reduce
    raise NotImplementedError()


# REFERENCE
def tensor_parallel_linear_ref(
    x: np.ndarray,
    W: np.ndarray,
    num_gpus: int,
    mode: str = 'column',
) -> np.ndarray:
    batch, in_features = x.shape
    in_feat, out_features = W.shape
    
    if mode == 'column':
        # Split output dimension
        chunk_size = out_features // num_gpus
        results = []
        for gpu in range(num_gpus):
            W_shard = W[:, gpu*chunk_size:(gpu+1)*chunk_size]
            results.append(x @ W_shard)  # (batch, chunk_size)
        return np.concatenate(results, axis=-1)  # all-gather
    
    elif mode == 'row':
        # Split input dimension
        chunk_size = in_features // num_gpus
        results = []
        for gpu in range(num_gpus):
            x_shard = x[:, gpu*chunk_size:(gpu+1)*chunk_size]
            W_shard = W[gpu*chunk_size:(gpu+1)*chunk_size, :]
            results.append(x_shard @ W_shard)  # (batch, out_features)
        return np.sum(results, axis=0)  # all-reduce sum


tensor_parallel_linear = tensor_parallel_linear_ref

# Verify tensor parallel gives same result as full computation
np.random.seed(42)
x = np.random.randn(4, 64)
W = np.random.randn(64, 128)

full_out = x @ W
tp_col = tensor_parallel_linear(x, W, num_gpus=4, mode='column')
tp_row = tensor_parallel_linear(x, W, num_gpus=4, mode='row')

print('Column parallel matches full:', np.allclose(full_out, tp_col))
print('Row parallel matches full:', np.allclose(full_out, tp_row))

---
## 8. Speculative Decoding

**Idea:** Use a small 'draft' model to generate K tokens quickly, then verify all K tokens in parallel with the large model.

**Win condition:** If the draft model guesses correctly, you get K tokens for the cost of ~1 forward pass.
**Guarantee:** Always produces the same distribution as the target model (rejection sampling).

In [ ]:
def speculative_decode_step(
    draft_probs: np.ndarray,      # (K, vocab) draft model probabilities
    target_probs: np.ndarray,     # (K, vocab) target model probabilities  
    draft_tokens: List[int],      # K tokens sampled from draft
) -> Tuple[List[int], int]:
    """
    Accept/reject draft tokens using speculative sampling.
    
    For each draft token t_i:
    - Accept with probability min(1, target_prob[t_i] / draft_prob[t_i])
    - If rejected, sample from adjusted distribution and stop
    
    Returns:
        accepted_tokens: list of accepted tokens (possibly empty)
        bonus_token: one more token sampled from target
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# REFERENCE
def speculative_decode_step_ref(
    draft_probs: np.ndarray,
    target_probs: np.ndarray,
    draft_tokens: List[int],
) -> Tuple[List[int], int]:
    accepted = []
    
    for i, token in enumerate(draft_tokens):
        # Acceptance probability
        p_draft = draft_probs[i, token]
        p_target = target_probs[i, token]
        accept_prob = min(1.0, p_target / (p_draft + 1e-10))
        
        if np.random.random() < accept_prob:
            accepted.append(token)
        else:
            # Rejected: sample from adjusted distribution
            adjusted = np.maximum(0, target_probs[i] - draft_probs[i])
            adjusted = adjusted / (adjusted.sum() + 1e-10)
            bonus = int(np.random.choice(len(adjusted), p=adjusted))
            return accepted, bonus
    
    # All accepted: sample one more from target distribution
    bonus = int(np.random.choice(target_probs.shape[1], p=target_probs[-1]))
    return accepted, bonus


speculative_decode_step = speculative_decode_step_ref

# Simulate speculative decoding
np.random.seed(42)
K = 4
vocab_size = 10

# Case 1: draft and target very similar (high acceptance)
base_probs = np.random.dirichlet(np.ones(vocab_size) * 2, size=K)
draft_p = base_probs * (1 + np.random.randn(K, vocab_size) * 0.01)
draft_p = np.abs(draft_p) / np.abs(draft_p).sum(axis=-1, keepdims=True)
target_p = base_probs

draft_toks = [np.argmax(draft_p[i]) for i in range(K)]
accepted, bonus = speculative_decode_step(draft_p, target_p, draft_toks)
print(f'K={K} draft tokens, accepted={len(accepted)}, +1 bonus = {len(accepted)+1} total')
print(f'Speedup estimate: {(len(accepted)+1)/1:.1f}x vs 1 token per target call')

---
## 9. System Design: Serving a 70B Model

**Prompt:** *Design an LLM inference server for Mixtral 8x7B (47B params) to handle 1000 concurrent requests with P95 latency < 5s.*

In [ ]:
def design_inference_cluster(
    model_params_b: float = 47.0,       # billion params
    dtype_bytes: float = 2.0,           # BF16
    gpu_memory_gb: float = 80.0,        # A100 80GB
    target_concurrent_reqs: int = 1000,
    avg_seq_len: int = 2048,
    target_p95_latency_s: float = 5.0,
    num_kv_heads: int = 8,
    head_dim: int = 128,
    num_layers: int = 32,
):
    """
    Calculate cluster requirements for LLM serving.
    """
    print(f'=== INFERENCE CLUSTER DESIGN ===')
    print(f'Model: {model_params_b}B params, BF16')
    print()
    
    # Model weight memory
    weight_gb = model_params_b * 1e9 * dtype_bytes / 1e9
    gpus_for_weights = math.ceil(weight_gb / (gpu_memory_gb * 0.7))  # 70% for weights
    print(f'1. MODEL WEIGHTS')
    print(f'   Size: {weight_gb:.0f} GB')
    print(f'   Minimum GPUs (TP): {gpus_for_weights}')
    
    # KV cache memory per request
    kv_per_req_mb = (2 * num_layers * num_kv_heads * head_dim * avg_seq_len * dtype_bytes) / 1e6
    print(f'\n2. KV CACHE')
    print(f'   Per request ({avg_seq_len} tokens): {kv_per_req_mb:.1f} MB')
    remaining_per_gpu = gpu_memory_gb * 0.3  # 30% for KV
    reqs_per_gpu = int(remaining_per_gpu * 1000 / kv_per_req_mb)
    print(f'   Max concurrent requests per {gpu_memory_gb}GB GPU (30% budget): {reqs_per_gpu}')
    
    # Replicas needed
    gpus_per_replica = gpus_for_weights
    reqs_per_replica = min(reqs_per_gpu, 64)  # practical limit
    num_replicas = math.ceil(target_concurrent_reqs / reqs_per_replica)
    total_gpus = num_replicas * gpus_per_replica
    
    print(f'\n3. CLUSTER SIZING')
    print(f'   GPUs per replica (tensor parallel): {gpus_per_replica}')
    print(f'   Concurrent requests per replica: {reqs_per_replica}')
    print(f'   Replicas needed for {target_concurrent_reqs} concurrent: {num_replicas}')
    print(f'   Total GPUs: {total_gpus} × A100-80GB')
    
    print(f'\n4. ARCHITECTURE')
    print(f'   - Load balancer (nginx/envoy) → {num_replicas} replicas')
    print(f'   - Each replica: {gpus_per_replica} GPUs with tensor parallelism')
    print(f'   - Continuous batching within each replica')
    print(f'   - Request queue with priority scheduling')
    print(f'   - Prefix cache for shared system prompts')
    print(f'   - Monitoring: TTFT, TPS, GPU util, queue depth')


design_inference_cluster()

---
## 10. Key Numbers to Know for Interview

Run this cell to review before the interview:

In [ ]:
print('=== CHEAT SHEET: LLM SERVING NUMBERS ===')
print()

print('MODEL MEMORY (weights only):')
for params_b, model in [(7.3, 'Mistral 7B'), (46.7, 'Mixtral 8x7B')]:
    for dtype, bpp in [('BF16', 2), ('INT8', 1), ('INT4', 0.5)]:
        gb = params_b * bpp
        print(f'  {model} {dtype}: {gb:.0f} GB')
    print()

print('KV CACHE (Mistral 7B: 32 layers, 8 KV heads, 128 head_dim, BF16):')
for seq_len in [1024, 4096, 16384, 32768]:
    mb = 2 * 32 * 8 * 128 * seq_len * 2 / 1e6
    print(f'  {seq_len:6d} tokens/req: {mb:.0f} MB')
print()

print('GPU SPECS (rough):')
for gpu, mem, tflops in [('RTX 4090', 24, 82), ('A100-40GB', 40, 77), ('A100-80GB', 80, 77), ('H100', 80, 267)]:
    print(f'  {gpu}: {mem}GB VRAM, {tflops} TFLOPS BF16')
print()

print('RULES OF THUMB:')
print('  1B params ≈ 2GB BF16')
print('  KV cache can be 30-50% of serving memory')
print('  Chinchilla: train on ~20 tokens per parameter')
print('  Continuous batching: ~3x better GPU utilization vs static')
print('  INT4 quantization: ~2x memory saving, ~5-10% quality loss')

---
## Interview Questions

1. **Memory:** A customer wants to run Mixtral 8x7B at INT4 on a single 24GB GPU. Is this possible?

2. **KV cache:** Why is the KV cache a bottleneck at long contexts? What does PagedAttention solve?

3. **Parallelism:** For a 70B model inference on 8 GPUs, would you use tensor or pipeline parallelism? Why? (Hint: pipeline has bubble overhead, tensor has communication overhead)

4. **Speculative decoding:** When does speculative decoding NOT help? (when draft quality is very low, or target model is fast enough)

5. **Latency vs throughput:** A customer reports P50 latency of 2s is fine, but P99 is 45s. What do you investigate?

6. **Flash Attention:** Explain why the attention computation is memory-bound, not compute-bound, and how FlashAttention fixes this.

---
**Next:** `06_advanced_moe_and_finetuning.ipynb`